   
# Approval Check — Epic on FHIR Model

Deployment job **approval task** (task name must start with 'approval').

Checks Unity Catalog tags for manual approval before promoting the model to
"champion" and updating the serving endpoint.

**Approval pattern (MLflow 3)**: The tag key **must match the job task name**
(`approval_check`). When an approver clicks "Approve" in the UC model version
UI, the system sets tag `approval_check = Approved` and auto-repairs the job run.

**Required job parameters**: `model_name`, `model_version`

This task **fails** if the approval tag is missing or not 'approved', which blocks
the downstream deployment task.

In [0]:
dbutils.widgets.text("model_name", "", "Model Name (catalog.schema.model)")
dbutils.widgets.text("model_version", "", "Model Version")
dbutils.widgets.text("approval_timeout_minutes", "30", "Approval Timeout (minutes)")

model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
approval_timeout_minutes = dbutils.widgets.get("approval_timeout_minutes")

assert model_name, "model_name parameter is required"
assert model_version, "model_version parameter is required"

print(f"Checking approval for {model_name} v{model_version}")
print(f"Timeout: {approval_timeout_minutes} minutes")

In [0]:
import time
from mlflow import MlflowClient

# MLflow 3 convention: the tag key MUST match the deployment job's approval
# task_key so the UC model-version UI "Approve" button and auto-repair work.
APPROVAL_TAG_KEY = "approval_check"

client = MlflowClient()
timeout_seconds = int(approval_timeout_minutes) * 60
poll_interval = 30
elapsed = 0

print(f"Waiting up to {approval_timeout_minutes} minutes for approval tag on {model_name} v{model_version}...")
print(f"Set tag: {APPROVAL_TAG_KEY} = approved")
print()

while elapsed < timeout_seconds:
    model_version_details = client.get_model_version(model_name, model_version)
    tags = model_version_details.tags or {}
    approval_tag = tags.get(APPROVAL_TAG_KEY, "")

    if approval_tag.lower() == "approved":
        print(f"\u2713 Model version {model_version} approved for deployment (after {elapsed}s)")
        break
    elif approval_tag.lower() == "rejected":
        raise ValueError(
            f"Model version {model_version} was explicitly rejected.\n"
            f"  Tag: {APPROVAL_TAG_KEY} = {approval_tag}\n"
            f"  To re-approve, update the tag to 'approved' and re-run the deployment job."
        )
    else:
        mins_remaining = (timeout_seconds - elapsed) // 60
        print(f"  [{elapsed}s] Waiting for approval... (tag={approval_tag!r}, {mins_remaining}m remaining)")
        time.sleep(poll_interval)
        elapsed += poll_interval
else:
    raise TimeoutError(
        f"Approval timeout after {approval_timeout_minutes} minutes.\n"
        f"To approve, set the UC tag on this model version:\n"
        f"  1. Unity Catalog UI: Browse to {model_name} v{model_version} > click 'Approve' in the deployment job sidebar\n"
        f"  2. API: MlflowClient().set_model_version_tag('{model_name}', '{model_version}', '{APPROVAL_TAG_KEY}', 'approved')\n"
        f"  3. CLI: databricks api post /api/2.0/mlflow/unity-catalog/model-versions/set-tag "
        f"--json '{{\"name\": \"{model_name}\", \"version\": \"{model_version}\", \"key\": \"{APPROVAL_TAG_KEY}\", \"value\": \"approved\"}}'"
    )

In [0]:
import json

dbutils.notebook.exit(json.dumps({
    "model_name": model_name,
    "model_version": model_version,
    "approval": "passed",
    "approval_tag": approval_tag,
}))